<a href="https://colab.research.google.com/github/Shahid-Ahmed-S/GenAI/blob/main/Sumarize.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U transformers



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 64.7 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.18.0
    Uninstalling transformers-5.18.0:
      Successfully uninstalled transformers-5.18.0


In [ ]:
!pip install transformers gradio sentencepiece -q
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers.utils import logging
import warnings
warnings.filterwarnings('ignore')
logging.set_verbosity_error()

MODEL_NAME = 'sshleifer/distilbart-cnn-12-6'

print('Loading model...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
print('Model ready!')


Loading model...


config.json:   0%|          | 0.00/1.80k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.22GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

Model ready!


In [ ]:
def summarize(text, max_words=60):
    # T5 needs 'summarize: ' at the start
    input_text = 'summarize: ' + text

    # Convert text to numbers
    inputs = tokenizer(
        input_text,
        return_tensors = 'pt',
        max_length     = 512,
        truncation     = True
    )

    # Generate the summary
    output = model.generate(
        **inputs,
        max_new_tokens  = max_words,
        min_length      = 10,
        length_penalty  = 2.0,
        num_beams       = 4,
        early_stopping  = True
    )

    # Convert numbers back to text
    result = tokenizer.decode(output[0], skip_special_tokens=True)
    return result

print('summarize() function ready!')


summarize() function ready!


In [ ]:
long_text = '''
Artificial intelligence is rapidly transforming industries around the world.
From healthcare to transportation, AI systems are being used to automate tasks,
improve decision making, and discover patterns in large datasets.
Hospitals are using AI to detect diseases earlier.
Self-driving cars use AI to navigate roads safely.
Farmers are using AI to predict weather and optimise crop yields.
The technology is growing fast and experts believe AI will change
nearly every job and industry within the next ten years.
'''

summary = summarize(long_text)

print('Original:')
print(long_text.strip())
print()
print('Summary:')
print(summary)
print()
print(f'Original : {len(long_text.split())} words')
print(f'Summary  : {len(summary.split())} words')
print(f'Reduced  : {round((1 - len(summary.split())/len(long_text.split()))*100)}%')

Original:
Artificial intelligence is rapidly transforming industries around the world.
From healthcare to transportation, AI systems are being used to automate tasks,
improve decision making, and discover patterns in large datasets.
Hospitals are using AI to detect diseases earlier.
Self-driving cars use AI to navigate roads safely.
Farmers are using AI to predict weather and optimise crop yields.
The technology is growing fast and experts believe AI will change
nearly every job and industry within the next ten years.

Summary:
 Artificial intelligence is rapidly transforming industries around the world . Experts believe AI will change nearly every job and industry within ten years .

Original : 78 words
Summary  : 24 words
Reduced  : 69%


In [ ]:
import gradio as gr

def summarize_app(text, max_words):
    if not text.strip():
        return 'Please paste some text first!'
    if len(text.split()) < 20:
        return 'Please enter at least 20 words!'

    summary    = summarize(text, max_words=int(max_words))
    orig_count = len(text.split())
    summ_count = len(summary.split())
    reduction  = round((1 - summ_count / orig_count) * 100)

    return f'{summary}\n\n--- Stats ---\nOriginal : {orig_count} words\nSummary  : {summ_count} words\nReduced  : {reduction}%'


app = gr.Interface(
    fn      = summarize_app,
    inputs  = [
        gr.Textbox(
            label       = 'Paste your long text here',
            placeholder = 'Paste any paragraph or article...',
            lines       = 6
        ),
        gr.Slider(
            minimum = 20,
            maximum = 80,
            value   = 50,
            step    = 10,
            label   = 'Summary length (words)'
        )
    ],
    outputs     = gr.Textbox(label='Summary + Stats', lines=6),
    title       = 'Summarize with AI',
    description = 'Paste any long text — AI shortens it for you!'
)

app.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b62fc15ac29647ad53.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
